In [77]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt


In [78]:
words = open('names.txt', 'r').read().splitlines()
len(words)

32033

In [79]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [80]:
block_size = 3
X, Y = [], []

for w in words[:5]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '---->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ----> e
..e ----> m
.em ----> m
emm ----> a
mma ----> .
olivia
... ----> o
..o ----> l
.ol ----> i
oli ----> v
liv ----> i
ivi ----> a
via ----> .
ava
... ----> a
..a ----> v
.av ----> a
ava ----> .
isabella
... ----> i
..i ----> s
.is ----> a
isa ----> b
sab ----> e
abe ----> l
bel ----> l
ell ----> a
lla ----> .
sophia
... ----> s
..s ----> o
.so ----> p
sop ----> h
oph ----> i
phi ----> a
hia ----> .


In [81]:
X.shape

torch.Size([32, 3])

In [82]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]])

In [83]:
C = torch.randn(27,2)

In [84]:
torch.set_printoptions(precision=4, sci_mode=False)
for idx in range(len(C)):
    print(C[idx])
    print(itos[idx])

tensor([ 1.5832, -0.3689])
.
tensor([1.0668, 0.8891])
a
tensor([1.0607, 0.6970])
b
tensor([-1.1069, -1.2177])
c
tensor([-0.0105,  0.9169])
d
tensor([ 0.0316, -1.0668])
e
tensor([1.1465, 0.3844])
f
tensor([1.6724, 0.7042])
g
tensor([0.3680, 1.5963])
h
tensor([ 0.1296, -0.8156])
i
tensor([0.5203, 1.3673])
j
tensor([0.0834, 1.0371])
k
tensor([1.2630, 0.0128])
l
tensor([-0.6677,  0.2841])
m
tensor([-1.2430,  0.6706])
n
tensor([-0.5623, -0.5297])
o
tensor([0.2680, 0.1149])
p
tensor([-2.1946, -1.3866])
q
tensor([-0.0691,  1.8946])
r
tensor([-0.8088,  0.3894])
s
tensor([-1.2249,  0.2849])
t
tensor([ 0.5785, -0.9166])
u
tensor([-1.3608,  1.6647])
v
tensor([2.8548, 0.7372])
w
tensor([1.1191, 0.3866])
x
tensor([ 0.9337, -0.8968])
y
tensor([2.0055, 1.5979])
z


In [85]:
C[X]

tensor([[[ 1.5832, -0.3689],
         [ 1.5832, -0.3689],
         [ 1.5832, -0.3689]],

        [[ 1.5832, -0.3689],
         [ 1.5832, -0.3689],
         [ 0.0316, -1.0668]],

        [[ 1.5832, -0.3689],
         [ 0.0316, -1.0668],
         [-0.6677,  0.2841]],

        [[ 0.0316, -1.0668],
         [-0.6677,  0.2841],
         [-0.6677,  0.2841]],

        [[-0.6677,  0.2841],
         [-0.6677,  0.2841],
         [ 1.0668,  0.8891]],

        [[ 1.5832, -0.3689],
         [ 1.5832, -0.3689],
         [ 1.5832, -0.3689]],

        [[ 1.5832, -0.3689],
         [ 1.5832, -0.3689],
         [-0.5623, -0.5297]],

        [[ 1.5832, -0.3689],
         [-0.5623, -0.5297],
         [ 1.2630,  0.0128]],

        [[-0.5623, -0.5297],
         [ 1.2630,  0.0128],
         [ 0.1296, -0.8156]],

        [[ 1.2630,  0.0128],
         [ 0.1296, -0.8156],
         [-1.3608,  1.6647]],

        [[ 0.1296, -0.8156],
         [-1.3608,  1.6647],
         [ 0.1296, -0.8156]],

        [[-1.3608,  1

In [86]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [87]:
W1 = torch.rand(6,100)
b1 = torch.rand(100)

In [88]:
torch.cat([emb[:,0,:], emb[:,1,:], emb[:,2,:]], 1).shape

torch.Size([32, 6])

In [89]:
torch.cat(torch.unbind(emb, 1),1).shape

torch.Size([32, 6])

In [90]:
h = (emb.view(-1,6)) @ W1 + b1
#h = (emb.view(32,6)) @ W1 + b1
#h = (emb.view(emb.shape[0],6)) @ W1 + b1

W1.shape == 32,100

B1.shape == 100

32,100

1 ,100

broadcastable because each column has either equal values, 1, or DNE value

(emb.view(32,6)) @ W1 + b1 == (emb.view(-1,6)) @ W1 + b1 == (emb.view(emb.shape[0],6)) @ W1 + b1


In [102]:
h = torch.tanh(h)

In [103]:
W2 = torch.rand(100,27)
B2 = torch.rand(27)

In [104]:
logits = h @ W2 + B2

In [105]:
logits.shape

torch.Size([32, 27])

In [106]:
counts = logits.exp()

In [107]:
prob = counts/counts.sum(1,keepdims=True)

In [108]:
prob.shape

torch.Size([32, 27])

In [109]:
prob

tensor([[0.4672, 0.0006, 0.0037, 0.3436, 0.0000, 0.0001, 0.0095, 0.0020, 0.0013,
         0.0001, 0.0356, 0.0001, 0.0292, 0.0024, 0.0207, 0.0046, 0.0001, 0.0003,
         0.0001, 0.0646, 0.0000, 0.0004, 0.0109, 0.0020, 0.0005, 0.0000, 0.0000],
        [0.7363, 0.0075, 0.0121, 0.0835, 0.0001, 0.0008, 0.0054, 0.0036, 0.0007,
         0.0003, 0.0081, 0.0002, 0.0171, 0.0044, 0.0390, 0.0154, 0.0007, 0.0004,
         0.0005, 0.0497, 0.0001, 0.0019, 0.0052, 0.0018, 0.0043, 0.0004, 0.0003],
        [0.2162, 0.0851, 0.0870, 0.0806, 0.0009, 0.0162, 0.1367, 0.0129, 0.0058,
         0.0020, 0.0153, 0.0025, 0.0115, 0.0153, 0.0079, 0.0156, 0.0404, 0.0018,
         0.0007, 0.0474, 0.0007, 0.0369, 0.0826, 0.0095, 0.0267, 0.0216, 0.0205],
        [0.0035, 0.0243, 0.0152, 0.0344, 0.0596, 0.0293, 0.0303, 0.0017, 0.0022,
         0.0062, 0.0038, 0.1199, 0.0091, 0.0081, 0.0024, 0.0009, 0.0154, 0.0034,
         0.0051, 0.0105, 0.0008, 0.0391, 0.0046, 0.0059, 0.1266, 0.3060, 0.1317],
        [0.0788, 0.0024,

In [110]:
prob[0].sum()

tensor(1.)